# 02 - Preprocesamiento

**Reorganizado el 26/09/2026** a partir de `02_desercion_eda_prep.ipynb`. Repite, sin cambios, las celdas de entorno/carga y las tres definiciones que la sección 5 (partición y pipelines) necesita y que se calcularon durante el EDA de `01_exploracion.ipynb` — grupos de columnas (`TARGET`/`FEAT_NUM`/`FEAT_CAT`), el filtro de censurados que produce `df_e`, y `BLOQUE_UMBRAL` (columna que participa en la definición de la etiqueta, usada más adelante en el análisis de sensibilidad del modelado). Se repiten aquí tal cual para que este notebook corra de forma independiente, sin depender de haber ejecutado antes `01_exploracion.ipynb` en la misma sesión de kernel.

In [1]:
# ====================== ENTORNO ======================
# La recolección (conexión a la base, SQL, caché) se hace en
# 01_desercion_dataset.ipynb. Aquí solo se prepara lo necesario para leer
# el parquet ya generado: no se abre ninguna conexión a la base.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 9})

DIR_DATOS = Path("datos");      DIR_DATOS.mkdir(exist_ok=True)
DIR_SALIDA = Path("salidas");   DIR_SALIDA.mkdir(exist_ok=True)
DIR_ART = Path("artefactos");   DIR_ART.mkdir(exist_ok=True)

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

# mismo interruptor que en 01_desercion_dataset.ipynb: decide qué parquet leer
USAR_BASE_REAL = True      # False = ds_desercion_sintetico.parquet · True = ds_desercion_real.parquet
print("modo:", "BASE REAL" if USAR_BASE_REAL else "SINTÉTICO")

# PARAMS y CORTES no hacen falta para leer el parquet (ya están "horneados"
# en las filas), pero 5.6 los guarda como metadata junto con el joblib de
# preprocesamiento, así que se redefinen aquí, idénticos a
# 01_desercion_dataset.ipynb (sin volver a tocar la base).
PARAMS = {
    "obs_meses":     12,
    "pred_dias":     120,
    "min_visitas":   4,
    "percentil":     0.90,
    "piso_dias":     14,
    "techo_dias":    180,
    "max_dias_anio":     150,
    "max_fact_por_dia":  3.0,
    "muestra_k":     10,
}
CORTES = [
    {"id": "C1", "fecha": "2024-12-31", "uso": "train"},
    {"id": "C2", "fecha": "2025-04-30", "uso": "train"},
    {"id": "C3", "fecha": "2025-08-31", "uso": "train"},
    {"id": "C4", "fecha": "2025-12-31", "uso": "validacion"},
    {"id": "C5", "fecha": "2026-04-30", "uso": "test"},
]

modo: BASE REAL


In [2]:
# ===================== CARGA DEL DATASET YA CONSTRUIDO =========================
RUTA_CACHE = DIR_DATOS / ("ds_desercion_real.parquet" if USAR_BASE_REAL
                          else "ds_desercion_sintetico.parquet")

if not RUTA_CACHE.exists():
    raise FileNotFoundError(
        f"No se encontró {RUTA_CACHE}. Corre primero 01_desercion_dataset.ipynb "
        "(con USAR_BASE_REAL en el mismo valor) para generar el parquet."
    )

df = pd.read_parquet(RUTA_CACHE)
print("leído:", RUTA_CACHE)
print(f"filas: {len(df):,}   columnas: {df.shape[1]}   "
      f"clientes únicos: {df['id_cliente'].nunique():,}")
df.head(3)

leído: datos\ds_desercion_real.parquet
filas: 2,417,698   columnas: 21   clientes únicos: 767,457


,fecha_corte,id_cliente,fecha_ultima_compra,recency_dias,n_visitas_12m,n_dias_activo_12m,monto_total_12m,ticket_promedio,tendencia_gasto,pct_gasto_cronico_12m,es_comprador_cronico,n_categorias_distintas_12m,n_bodegas_distintas_12m,antiguedad_meses,metodo_pago_principal_12m,n_metodos_pago_distintos_12m,umbral_propio_dias,censurado,churn,corte_id,uso
0,2024-12-31,1729,2024-11-15,46,5,5,47.3900,9.4780,8.9975,0.0000,False,3,4,13,EFE,1,157,1,NaN,C1,train
1,2024-12-31,1743,2024-11-06,55,4,4,7.8000,1.9500,0.5135,0.0000,False,4,3,13,EFE,1,105,0,0.0000,C1,train
2,2024-12-31,1812,2024-06-15,199,22,17,292.9000,13.3136,0.0000,0.0000,False,5,3,13,EFE,2,22,0,1.0000,C1,train


In [3]:
# ===================== GRUPOS DE COLUMNAS (esquema de 16 columnas, 20/09/2026) =========================
# segmento_frecuencia ya no viene del SQL (el diseño simplificado no la calcula).
# Se reconstruye aquí con los mismos tramos medidos en CONTEXTO_PROYECTO.md
# (sección 2.2): baja 4-6 visitas/año, media 7-12, alta 13+. El SQL ya filtra
# n_visitas_12m >= PARAMS["min_visitas"] (4), así que no hay tramo por debajo.
def _segmentar(n):
    if n <= 6:
        return "1. baja"
    elif n <= 12:
        return "2. media"
    return "3. alta"

df["segmento_frecuencia"] = df["n_visitas_12m"].apply(_segmentar)

# SQL Server CAST(...AS bit) llega como bool de Python/pandas; describe(),
# .quantile() y .skew() no operan sobre bool (numpy no permite restarlos),
# así que se castea a entero apenas se define como feature numérica.
df["es_comprador_cronico"] = df["es_comprador_cronico"].astype(int)

COLS_CONTROL = ["id_cliente", "fecha_corte", "fecha_ultima_compra", "corte_id",
                "uso", "segmento_frecuencia", "censurado"]
COLS_ETIQUETA = ["churn"]          # única etiqueta en el esquema simplificado
TARGET = "churn"

FEAT_NUM = [
    # ritmo y actividad
    "recency_dias", "n_visitas_12m", "n_dias_activo_12m",
    # monetario
    "monto_total_12m", "ticket_promedio", "tendencia_gasto",
    # canasta farmacéutica
    "pct_gasto_cronico_12m", "es_comprador_cronico",
    # variedad y antigüedad
    "n_categorias_distintas_12m", "n_bodegas_distintas_12m", "antiguedad_meses",
    # pago
    "n_metodos_pago_distintos_12m",
    # umbral propio (participa de la etiqueta, ver 4.11)
    "umbral_propio_dias",
]
FEAT_CAT = ["metodo_pago_principal_12m"]   # única categórica que sobrevive al SQL simplificado
FEATURES = FEAT_NUM + FEAT_CAT

faltan = [c for c in COLS_CONTROL + COLS_ETIQUETA + FEATURES if c not in df.columns]
sobran = [c for c in df.columns if c not in COLS_CONTROL + COLS_ETIQUETA + FEATURES]
print(f"features numéricas : {len(FEAT_NUM)}")
print(f"features categóricas: {len(FEAT_CAT)}")
print(f"total features      : {len(FEATURES)}")
if faltan: print("FALTAN en el dataframe:", faltan)
if sobran: print("columnas no clasificadas:", sobran)

features numéricas : 13
features categóricas: 1
total features      : 14


In [6]:
cen = (df.groupby("segmento_frecuencia", observed=True)
         .agg(clientes=("censurado", "size"),
              censurados=("censurado", "sum"),
              umbral_medio=("umbral_propio_dias", "mean")))
cen["pct_censura"] = (100*cen["censurados"]/cen["clientes"]).round(2)
cen["entrenables"] = cen["clientes"] - cen["censurados"]
print(cen.round(2).to_string())
print(f"\ncensura global ponderada: "
      f"{100*df['censurado'].mean():.2f} %")

# ---- a partir de aquí se trabaja solo con los entrenables ----
df_e = df.loc[df["censurado"] == 0].copy()

# caso límite: no censurado pero sin umbral propio calculable (menos de 2
# fechas distintas de compra en la ventana, así que no hay intervalo que
# percentilar). El SQL los deja con churn NULL; se reportan y se excluyen.
nulos_churn = int(df_e["churn"].isna().sum())
if nulos_churn:
    print(f"\nclientes no censurados con churn nulo (sin umbral propio "
          f"calculable): {nulos_churn} — se excluyen del set entrenable")
    df_e = df_e.loc[df_e["churn"].notna()].copy()
df_e["churn"] = df_e["churn"].astype(int)

print(f"\ndataset entrenable: {len(df_e):,} filas "
      f"({100*len(df_e)/len(df):.1f} % del total), "
      f"{df_e['id_cliente'].nunique():,} clientes únicos")

                     clientes  censurados  umbral_medio  pct_censura  entrenables
segmento_frecuencia                                                              
1. baja                999564      335243       98.8300      33.5400       664321
2. media               707612       46832       74.2000       6.6200       660780
3. alta                710522         423       35.2300       0.0600       710099

censura global ponderada: 15.82 %

dataset entrenable: 2,035,200 filas (84.2 % del total), 682,128 clientes únicos


In [17]:
print("— 1. columnas de etiqueta fuera de features —")
prohibidas = COLS_ETIQUETA + ["censurado"]
intrusas = [c for c in prohibidas if c in FEATURES]
print("OK: ninguna" if not intrusas else f"ERROR: {intrusas}")

print("\n— 2. bloque ligado a la definición de la etiqueta —")
BLOQUE_UMBRAL = ["umbral_propio_dias"]
print("a excluir en el análisis de sensibilidad del modelado:", BLOQUE_UMBRAL)
print("(en el diseño anterior este bloque tenía 3 columnas — gap_medio_dias,"
      " umbral_dias, ratio_recency_umbral —; el SQL simplificado solo"
      " conserva el equivalente de umbral_dias)")

— 1. columnas de etiqueta fuera de features —
OK: ninguna

— 2. bloque ligado a la definición de la etiqueta —
a excluir en el análisis de sensibilidad del modelado: ['umbral_propio_dias']
(en el diseño anterior este bloque tenía 3 columnas — gap_medio_dias, umbral_dias, ratio_recency_umbral —; el SQL simplificado solo conserva el equivalente de umbral_dias)


---
## Reconstrucción de variables usadas por la sección 5
(idénticas a `01_exploracion.ipynb`, secciones 4.1/4.3/4.11)

---
# 5. Codificación y normalización

## 5.1 Partición

Dos reglas que se aplican **en conjunto**:

**Out-of-time.** El test es siempre el corte más reciente (C5) y la validación el
anterior (C4). Nunca se mezclan cortes al azar: hacerlo equivale a entrenar con el
futuro.

**Por cliente, no por fila.** Un mismo cliente aparece en varios cortes y sus
filas están correlacionadas —es la misma persona, con el mismo comportamiento de
base. Si aparece en train y en test, el modelo lo «reconoce» y las métricas salen
infladas.

Combinarlas cuesta filas: se asigna cada cliente a un grupo (70/15/15) y luego se
intersecta con el corte que le corresponde. Un cliente del grupo de train que solo
aparece en C5 se pierde. Es el precio de una evaluación limpia, y con este volumen
se puede pagar.

In [18]:
from sklearn.model_selection import GroupShuffleSplit

# 1) asignación de cada CLIENTE a un grupo, de forma determinista
clientes = np.sort(df_e["id_cliente"].unique())
r = np.random.default_rng(SEMILLA)
mezcla = r.permutation(len(clientes))
n_tr = int(.70*len(clientes)); n_va = int(.15*len(clientes))
grupo = pd.Series("test", index=clientes[mezcla], dtype="object")
grupo.iloc[:n_tr] = "train"
grupo.iloc[n_tr:n_tr+n_va] = "validacion"
grupo = grupo.sort_index()

df_e["grupo_cliente"] = df_e["id_cliente"].map(grupo)

# 2) intersección con el esquema temporal
CORTES_TRAIN = ["C1", "C2", "C3"]; CORTE_VAL = "C4"; CORTE_TEST = "C5"
cond_tr = df_e["corte_id"].isin(CORTES_TRAIN) & (df_e["grupo_cliente"] == "train")
cond_va = (df_e["corte_id"] == CORTE_VAL)  & (df_e["grupo_cliente"] == "validacion")
cond_te = (df_e["corte_id"] == CORTE_TEST) & (df_e["grupo_cliente"] == "test")

df_e["split"] = np.select([cond_tr, cond_va, cond_te],
                          ["train", "validacion", "test"], default="descartado")

resumen = (df_e.groupby("split", observed=True)
             .agg(filas=("id_cliente","size"), clientes=("id_cliente","nunique"),
                  tasa_base=(TARGET,"mean")))
resumen["tasa_base_%"] = (100*resumen["tasa_base"]).round(2)
print(resumen.drop(columns="tasa_base").to_string())

# 3) verificación: sin solapamiento de clientes entre particiones
ids = {s: set(df_e.loc[df_e["split"]==s, "id_cliente"])
       for s in ("train","validacion","test")}
print("\nsolapamiento train∩test      :", len(ids["train"] & ids["test"]))
print("solapamiento train∩validación:", len(ids["train"] & ids["validacion"]))
print("solapamiento val∩test        :", len(ids["validacion"] & ids["test"]))
assert not (ids["train"] & ids["test"]), "hay clientes compartidos entre train y test"
print("\nOK: particiones disjuntas por cliente")

              filas  clientes  tasa_base_%
split                                     
descartado  1104571    571340      39.8700
test          66421     66421      40.1500
train        796044    383549      37.3400
validacion    68164     68164      41.7100

solapamiento train∩test      : 0
solapamiento train∩validación: 0
solapamiento val∩test        : 0

OK: particiones disjuntas por cliente


In [19]:
tr = df_e[df_e["split"] == "train"].copy()
va = df_e[df_e["split"] == "validacion"].copy()
te = df_e[df_e["split"] == "test"].copy()

y_tr, y_va, y_te = tr[TARGET].to_numpy(), va[TARGET].to_numpy(), te[TARGET].to_numpy()
print(f"train      : {len(tr):>8,} filas · tasa base {100*y_tr.mean():.2f} %")
print(f"validación : {len(va):>8,} filas · tasa base {100*y_va.mean():.2f} %")
print(f"test       : {len(te):>8,} filas · tasa base {100*y_te.mean():.2f} %")
print(f"descartado : {int((df_e['split']=='descartado').sum()):>8,} filas")

train      :  796,044 filas · tasa base 37.34 %
validación :   68,164 filas · tasa base 41.71 %
test       :   66,421 filas · tasa base 40.15 %
descartado : 1,104,571 filas


## 5.2 Agrupación de categorías poco frecuentes

Las categorías que aparecen en menos del `MIN_FREQ` de los clientes de **train** se
colapsan en `OTROS`. El umbral se aprende solo con train: si se calculara sobre
todo el dataset, la composición del test influiría en la transformación.

Los nulos de `metodo_pago_principal_12m` (única categórica del esquema
simplificado) se codifican como categoría explícita `DESCONOCIDO` en lugar de
imputarse. Que falte el dato puede ser informativo, y es preferible dejarlo
visible antes que inventar un valor.

In [20]:
from sklearn.base import BaseEstimator, TransformerMixin

MIN_FREQ = 0.01          # 1 % de los clientes de train
ETIQ_RARA = "OTROS"
ETIQ_NULA = "DESCONOCIDO"

class AgrupadorRaras(BaseEstimator, TransformerMixin):
    """Colapsa en OTROS las categorías con frecuencia < min_freq en el fit."""
    def __init__(self, min_freq=0.01, etiq_rara=ETIQ_RARA, etiq_nula=ETIQ_NULA):
        self.min_freq, self.etiq_rara, self.etiq_nula = min_freq, etiq_rara, etiq_nula

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.columnas_ = list(X.columns)
        self.mantener_ = {}
        for c in self.columnas_:
            s = X[c].astype("object").where(X[c].notna(), self.etiq_nula).astype(str)
            fr = s.value_counts(normalize=True)
            mant = set(fr[fr >= self.min_freq].index)
            if self.etiq_nula in fr.index:      # solo si hay nulos de verdad
                mant.add(self.etiq_nula)
            self.mantener_[c] = mant
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.columnas_:
            s = X[c].astype("object").where(X[c].notna(), self.etiq_nula).astype(str)
            X[c] = np.where(s.isin(self.mantener_[c]), s, self.etiq_rara)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.columnas_, dtype=object)


agr = AgrupadorRaras(min_freq=MIN_FREQ).fit(tr[FEAT_CAT])
for c in FEAT_CAT:
    conserv = len(agr.mantener_[c] - {ETIQ_NULA})
    total   = df_e[c].nunique(dropna=True)
    print(f"{c:24s}: {total:>4} categorías → {conserv:>3} conservadas"
          f"{' + OTROS' if conserv < total else ' (ninguna rara)'}"
          f"{' + DESCONOCIDO' if ETIQ_NULA in agr.mantener_[c] else ''}")

cat_tr = agr.transform(tr[FEAT_CAT])
print("\ndistribución tras agrupar (train):")
for c in FEAT_CAT:
    vc = cat_tr[c].value_counts(normalize=True).mul(100).round(1)
    print(f"  {c}: {dict(vc.head(6))}")

metodo_pago_principal_12m:   14 categorías →   4 conservadas + OTROS

distribución tras agrupar (train):
  metodo_pago_principal_12m: {'EFE': np.float64(91.6), 'DEB': np.float64(3.2), 'CRP': np.float64(2.4), 'TAR': np.float64(2.2), 'OTROS': np.float64(0.5)}


## 5.3 Pipeline A — modelos de árbol

Los árboles parten por umbrales, así que la escala no importa: un corte en
`recency > 45` funciona igual con la variable en días o normalizada. Tampoco
necesitan imputación, porque LightGBM y XGBoost manejan los nulos de forma nativa
enviándolos a la rama que minimiza la pérdida.

Se construyen **dos variantes** para poder comparar, como se decidió:

- **A1 · one-hot.** Categóricas agrupadas y expandidas a columnas binarias. Es la
  base común con el pipeline B.
- **A2 · categóricas nativas.** Las mismas columnas marcadas como `category`, que
  LightGBM particiona directamente sin expandirlas.

La comparación de PR-AUC entre A1 y A2 es un resultado documentado, no una
elección arbitraria.

In [21]:
from sklearn.preprocessing import OneHotEncoder

# ---------- A1: numéricas en crudo + one-hot de las agrupadas ----------
ohe_arbol = OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                          dtype=np.float32).fit(agr.transform(tr[FEAT_CAT]))
nombres_ohe = list(ohe_arbol.get_feature_names_out(FEAT_CAT))

def preparar_A1(d):
    num = d[FEAT_NUM].replace([np.inf, -np.inf], np.nan).astype("float32")
    cat = pd.DataFrame(ohe_arbol.transform(agr.transform(d[FEAT_CAT])),
                       columns=nombres_ohe, index=d.index)
    return pd.concat([num, cat], axis=1)

# ---------- A2: numéricas en crudo + categóricas nativas ----------
CATEGORIAS = {c: sorted(set(agr.mantener_[c]) | {ETIQ_RARA}) for c in FEAT_CAT}

def preparar_A2(d):
    num = d[FEAT_NUM].replace([np.inf, -np.inf], np.nan).astype("float32")
    cat = agr.transform(d[FEAT_CAT])
    for c in FEAT_CAT:
        cat[c] = pd.Categorical(cat[c], categories=CATEGORIAS[c])
    cat.index = d.index
    return pd.concat([num, cat], axis=1)

XA1_tr, XA1_va, XA1_te = preparar_A1(tr), preparar_A1(va), preparar_A1(te)
XA2_tr, XA2_va, XA2_te = preparar_A2(tr), preparar_A2(va), preparar_A2(te)

print(f"A1 one-hot  : {XA1_tr.shape[1]} columnas "
      f"({len(FEAT_NUM)} numéricas + {len(nombres_ohe)} binarias)")
print(f"A2 nativas  : {XA2_tr.shape[1]} columnas "
      f"({len(FEAT_NUM)} numéricas + {len(FEAT_CAT)} categóricas)")
print("\nA2 dtypes de las categóricas:")
print(XA2_tr[FEAT_CAT].dtypes.to_string())

A1 one-hot  : 18 columnas (13 numéricas + 5 binarias)
A2 nativas  : 14 columnas (13 numéricas + 1 categóricas)

A2 dtypes de las categóricas:
metodo_pago_principal_12m    category


## 5.4 Pipeline B — regresión logística

Aquí cada transformación es necesaria, y por un motivo concreto:

| Paso | Por qué |
|---|---|
| Imputar con la mediana | La logística no admite nulos |
| Acotar al 1–99 % (*winsorizing*) | Un outlier arrastra los coeficientes por mínimos cuadrados |
| Logaritmo con signo en las asimétricas | Linealiza la relación y reduce el peso de la cola |
| Estandarizar | Sin esto, `monto_total_12m` (miles) domina a `pct_gasto_cronico_12m` (décimas) por escala, no por importancia |
| One-hot | La logística no consume categorías como texto |

Todos los parámetros —mediana, percentiles de recorte, media y desviación— se
aprenden **solo en train** y se aplican tal cual a validación y test. Calcularlos
sobre el conjunto completo sería fuga de información.

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Se usan CLASES en lugar de FunctionTransformer con lambda por dos razones:
# las lambdas no son serializables con joblib, y FunctionTransformer no expone
# get_feature_names_out, lo que rompe el ColumnTransformer.

class _PasaNombres:
    """Mixin: la transformación es 1 a 1, así que los nombres no cambian."""
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.asarray(getattr(self, "nombres_", []), dtype=object)
        return np.asarray(input_features, dtype=object)

class LimpiaInfinitos(_PasaNombres, BaseEstimator, TransformerMixin):
    """Convierte ±inf en nulo para que el imputador los trate."""
    def fit(self, X, y=None):
        self.nombres_ = list(getattr(X, "columns", []))
        return self
    def transform(self, X):
        A = np.asarray(X, dtype="float64")
        return np.where(np.isinf(A), np.nan, A)

class Winsorizador(_PasaNombres, BaseEstimator, TransformerMixin):
    """Acota cada columna a los percentiles aprendidos en el fit (solo train)."""
    def __init__(self, inferior=0.01, superior=0.99):
        self.inferior, self.superior = inferior, superior
    def fit(self, X, y=None):
        self.nombres_ = list(getattr(X, "columns", []))
        A = np.asarray(X, dtype="float64")
        self.lo_ = np.nanquantile(A, self.inferior, axis=0)
        self.hi_ = np.nanquantile(A, self.superior, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype="float64"), self.lo_, self.hi_)

class LogConSigno(_PasaNombres, BaseEstimator, TransformerMixin):
    """sign(x)·log(1+|x|): linealiza colas y admite valores negativos."""
    def fit(self, X, y=None):
        self.nombres_ = list(getattr(X, "columns", []))
        return self
    def transform(self, X):
        A = np.asarray(X, dtype="float64")
        return np.sign(A) * np.log1p(np.abs(A))

COLS_NUM_LOG  = [c for c in COLS_LOG if c in FEAT_NUM]
COLS_NUM_PLAN = [c for c in FEAT_NUM if c not in COLS_NUM_LOG]

pipe_log = Pipeline([
    ("limpia", LimpiaInfinitos()),
    ("imputa", SimpleImputer(strategy="median")),
    ("acota",  Winsorizador(0.01, 0.99)),
    ("log",    LogConSigno()),
    ("escala", StandardScaler()),
])
pipe_plano = Pipeline([
    ("limpia", LimpiaInfinitos()),
    ("imputa", SimpleImputer(strategy="median")),
    ("acota",  Winsorizador(0.01, 0.99)),
    ("escala", StandardScaler()),
])
pipe_cat = Pipeline([
    ("agrupa", AgrupadorRaras(min_freq=MIN_FREQ)),
    ("ohe",    OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                             dtype=np.float32)),
])

prep_B = ColumnTransformer([
    ("num_log",   pipe_log,   COLS_NUM_LOG),
    ("num_plano", pipe_plano, COLS_NUM_PLAN),
    ("cat",       pipe_cat,   FEAT_CAT),
], remainder="drop", verbose_feature_names_out=False)

prep_B.fit(tr[FEATURES], y_tr)
nombres_B = list(prep_B.get_feature_names_out())

XB_tr = pd.DataFrame(prep_B.transform(tr[FEATURES]), columns=nombres_B, index=tr.index)
XB_va = pd.DataFrame(prep_B.transform(va[FEATURES]), columns=nombres_B, index=va.index)
XB_te = pd.DataFrame(prep_B.transform(te[FEATURES]), columns=nombres_B, index=te.index)

print(f"B escalado: {XB_tr.shape[1]} columnas")
print(f"  {len(COLS_NUM_LOG)} numéricas con log + {len(COLS_NUM_PLAN)} numéricas planas "
      f"+ {XB_tr.shape[1]-len(FEAT_NUM)} binarias")
print("\ncomprobación de estandarización en train (debe ser media≈0, sd≈1):")
chk = XB_tr[(COLS_NUM_LOG[:3] + COLS_NUM_PLAN[:3])[:6]].agg(["mean", "std"]).T
print(chk.round(4).to_string())

B escalado: 18 columnas
  8 numéricas con log + 5 numéricas planas + 5 binarias

comprobación de estandarización en train (debe ser media≈0, sd≈1):
                              mean    std
n_visitas_12m               0.0000 1.0000
n_dias_activo_12m          -0.0000 1.0000
monto_total_12m             0.0000 1.0000
recency_dias                0.0000 1.0000
es_comprador_cronico       -0.0000 1.0000
n_categorias_distintas_12m -0.0000 1.0000


## 5.5 Verificaciones finales

Cinco comprobaciones antes de dar el dataset por listo. Si alguna falla, el
modelado partiría de datos defectuosos.

In [23]:
errores = []

# 1. sin nulos ni infinitos en el pipeline de la logística
for nombre, X in [("XB_tr", XB_tr), ("XB_va", XB_va), ("XB_te", XB_te)]:
    n_na = int(X.isna().sum().sum())
    n_inf = int(np.isinf(X.to_numpy()).sum())
    if n_na or n_inf:
        errores.append(f"{nombre}: {n_na} nulos, {n_inf} infinitos")
print("1. pipeline B sin nulos/infinitos:", "OK" if not errores else errores[-1])

# 2. mismas columnas y mismo orden en las tres particiones
ok_cols = (list(XB_tr.columns) == list(XB_va.columns) == list(XB_te.columns)
           and list(XA1_tr.columns) == list(XA1_te.columns))
print("2. columnas alineadas entre particiones:", "OK" if ok_cols else "ERROR")
if not ok_cols: errores.append("columnas desalineadas")

# 3. ninguna etiqueta ni columna de control dentro de las features
fuga = [c for c in COLS_ETIQUETA + ["censurado", "split", "grupo_cliente"]
        if c in FEATURES or c in XB_tr.columns or c in XA1_tr.columns]
print("3. sin etiquetas dentro de X:", "OK" if not fuga else f"ERROR {fuga}")
if fuga: errores.append(f"fuga de etiquetas: {fuga}")

# 4. filas coherentes
ok_filas = (len(XB_tr) == len(y_tr) and len(XB_va) == len(y_va)
            and len(XB_te) == len(y_te))
print("4. filas de X e y coinciden:", "OK" if ok_filas else "ERROR")
if not ok_filas: errores.append("desalineación X/y")

# 5. clientes disjuntos
ok_disj = not (ids["train"] & ids["test"]) and not (ids["train"] & ids["validacion"])
print("5. clientes disjuntos entre particiones:", "OK" if ok_disj else "ERROR")
if not ok_disj: errores.append("clientes compartidos")

print("\n" + ("TODAS LAS VERIFICACIONES PASARON"
               if not errores else f"REVISAR: {errores}"))

1. pipeline B sin nulos/infinitos: OK
2. columnas alineadas entre particiones: OK
3. sin etiquetas dentro de X: OK
4. filas de X e y coinciden: OK
5. clientes disjuntos entre particiones: OK

TODAS LAS VERIFICACIONES PASARON


## 5.6 Guardar artefactos

Se guardan los tres conjuntos preparados y los transformadores ajustados. Los
transformadores son imprescindibles: en producción hay que aplicar **exactamente**
la misma transformación (las mismas medianas, los mismos percentiles de recorte,
las mismas categorías conservadas). Reajustarlos sobre datos nuevos daría
resultados distintos.

In [24]:
import joblib

for nombre, obj in [("XA1_train", XA1_tr), ("XA1_val", XA1_va), ("XA1_test", XA1_te),
                    ("XA2_train", XA2_tr), ("XA2_val", XA2_va), ("XA2_test", XA2_te),
                    ("XB_train",  XB_tr),  ("XB_val",  XB_va),  ("XB_test",  XB_te)]:
    obj.to_parquet(DIR_DATOS / f"{nombre}.parquet", index=False)

pd.DataFrame({"y_train": y_tr}).to_parquet(DIR_DATOS / "y_train.parquet", index=False)
pd.DataFrame({"y_val":   y_va}).to_parquet(DIR_DATOS / "y_val.parquet",   index=False)
pd.DataFrame({"y_test":  y_te}).to_parquet(DIR_DATOS / "y_test.parquet",  index=False)

# el dataset entrenable con sus etiquetas y particiones, para el notebook de modelado
cols_guardar = (COLS_CONTROL + COLS_ETIQUETA + FEATURES
                + ["grupo_cliente", "split"])
df_e[cols_guardar].to_parquet(DIR_DATOS / "ds_entrenable.parquet", index=False)

joblib.dump({
    "prep_B": prep_B, "agrupador": agr, "ohe_arbol": ohe_arbol,
    "categorias": CATEGORIAS, "cols_num_log": COLS_NUM_LOG,
    "cols_num_plano": COLS_NUM_PLAN, "feat_num": FEAT_NUM, "feat_cat": FEAT_CAT,
    "bloque_umbral": BLOQUE_UMBRAL, "params": PARAMS, "cortes": CORTES,
    "min_freq": MIN_FREQ, "semilla": SEMILLA,
}, DIR_ART / "preprocesamiento.joblib")

print("guardado en", DIR_DATOS.resolve())
print()
print("AVISO: el joblib referencia las clases AgrupadorRaras, LimpiaInfinitos,")
print("Winsorizador y LogConSigno. Para cargarlo hay que ejecutar antes las celdas")
print("que las definen (5.2 y 5.4), o moverlas a un módulo .py importable.")
for p in sorted(DIR_DATOS.glob("*.parquet")):
    print(f"  {p.name:22s} {p.stat().st_size/1e6:>7,.1f} MB")
print("\nartefactos:", (DIR_ART / "preprocesamiento.joblib").resolve())

guardado en C:\Users\henry.medina\OneDrive\Escritorio\INTELIGENCIA_ARTIFICIAL\datos

AVISO: el joblib referencia las clases AgrupadorRaras, LimpiaInfinitos,
Winsorizador y LogConSigno. Para cargarlo hay que ejecutar antes las celdas
que las definen (5.2 y 5.4), o moverlas a un módulo .py importable.
  ds_desercion_real.parquet    64.5 MB
  ds_desercion_sintetico.parquet     7.2 MB
  ds_entrenable.parquet     55.3 MB
  XA1_test.parquet           1.3 MB
  XA1_train.parquet         13.2 MB
  XA1_val.parquet            1.4 MB
  XA2_test.parquet           1.3 MB
  XA2_train.parquet         13.2 MB
  XA2_val.parquet            1.3 MB
  XB_test.parquet            1.7 MB
  XB_train.parquet          16.8 MB
  XB_val.parquet             1.8 MB
  y_test.parquet             0.0 MB
  y_train.parquet            0.1 MB
  y_val.parquet              0.0 MB

artefactos: C:\Users\henry.medina\OneDrive\Escritorio\INTELIGENCIA_ARTIFICIAL\artefactos\preprocesamiento.joblib


---
## Estado y siguiente paso

Queda listo:

- **A1** numéricas en crudo + one-hot → árboles, base comparable con B
- **A2** numéricas en crudo + categóricas nativas → LightGBM, para la comparación
- **B** imputado, acotado, log-transformado, estandarizado y one-hot → regresión logística

### Antes de ejecutar sobre la base real

1. Correr la auto-detección (3.2, en `01_desercion_dataset.ipynb`) y completar
   `ESQUEMA` con los valores reales de `estado`, `tipo_transaccion`,
   `producto_cronico` y `generico`.
2. Verificar que `pct_gasto_cronico_12m` **no** salga en cero para todos. Si
   sale, el valor de la bandera no era el asumido.
3. Revisar la estabilidad de la tasa base entre cortes (4.2) antes de continuar.

### El notebook de modelado

Sigue la escalera de cuatro niveles, donde cada nivel debe ganarse su lugar
superando al anterior. **El nivel 0 cambia respecto al diseño original:** ya
no hay una columna `y_fijo90` en el dataset (el SQL simplificado no la calcula,
ver 4.10), así que el nivel 0 se define ahora como una regla de negocio sobre
`recency_dias` al momento de correr la campaña (p. ej. "avisar si
`recency_dias > 90`"), no como una etiqueta ya calculada sobre la ventana de
predicción.

| Nivel | Modelo | Rol |
|---|---|---|
| 0 | Regla de negocio: `recency_dias > 90` al momento de la campaña | Lo que la empresa haría hoy, sin IA |
| 1 | Segmentación RFM | Lo que ya está en producción |
| 2 | Regresión logística (pipeline B) | Interpretable y calibrada |
| 3 | LightGBM (A1 y A2) | Modelo principal |
| 4 | Análisis de supervivencia | Aprovecha los censurados en lugar de descartarlos |

**Si el nivel 3 no supera con claridad al nivel 1 en los primeros deciles, la
conclusión es que no hacía falta IA.** Eso es un hallazgo válido, y es mucho mejor
descubrirlo pronto que en la defensa.

Métricas: PR-AUC y precision@k con *k* igual al tamaño real de la campaña.
Nunca *accuracy*. Reportar siempre desglosado por segmento de frecuencia y por
`es_comprador_cronico`. Calibrar las probabilidades si se van a usar umbrales de
decisión, y explicar con SHAP: marketing no adopta una lista que no entiende.